In [1]:

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import os
import gc

In [2]:
adapter_path = "meta-llama_Llama-3.1-8B-Instruct_lora_epochs_10_batch_3_optim_adamw_torch_fused_r_16_alpha_32_q_proj_v_proj_k_proj_o_proj_gate_proj_up_proj_down_proj_embed_tokens_lm_head"
adapter_path= "../meta-llama_Llama-3.1-8B-Instruct_lora_epochs_10_batch_4_optim_adamw_torch_fused_r_64_alpha_128_lr_0.0002_drop_out_0.05_q_proj_v_proj_k_proj_o_proj_gate_proj_up_proj_down_proj_embed_tokens_lm_head"
adapter_path = "../ruslanmv_Medical-Llama3-8B_lora_epochs_10_batch_2_optim_adamw_torch_fused_r_16_alpha_32_q_proj_v_proj_k_proj_o_proj_gate_proj_up_proj_down_proj_embed_tokens_lm_head"
print(os.listdir(adapter_path))

['tokenizer.json', 'adapter_config.json', 'README.md', 'tokenizer_config.json', 'adapter_model.safetensors']


In [3]:
#ruslanmv/Medical-Llama3-8B
#ruslandev/llama-3-8b-gpt-4o-ru1.0
#meta-llama/Llama-3.1-8B-Instruct


base_model_name = "ruslanmv/Medical-Llama3-8B"
print("Загрузка базовой модели...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(base_model_name, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

Загрузка базовой модели...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [4]:
# Загрузка адаптера поверх базовой модели
print("Загрузка адаптера LoRA...")
model = PeftModel.from_pretrained(base_model, adapter_path)

Загрузка адаптера LoRA...


In [5]:
# Слияние адаптера с базовой моделью
print("Слияние адаптера...")
merged_model = model.merge_and_unload()

Слияние адаптера...


In [6]:
# Сохранение объединённой модели
output_dir = "LORA_THE_BEST"
print(f"Сохранение объединённой модели в {output_dir}...")
merged_model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

Сохранение объединённой модели в LORA_THE_BEST...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('LORA_THE_BEST/tokenizer_config.json', 'LORA_THE_BEST/tokenizer.json')

In [7]:
del merged_model
del model
del base_model
del tokenizer
gc.collect()
torch.cuda.empty_cache()
print(torch.cuda.memory_summary())

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   8320 KiB |  19508 MiB |  95349 MiB |  95341 MiB |
|       from large pool |   8320 KiB |  19347 MiB |  94947 MiB |  94939 MiB |
|       from small pool |      0 KiB |    321 MiB |    401 MiB |    401 MiB |
|---------------------------------------------------------------------------|
| Active memory         |   8320 KiB |  19508 MiB |  95349 MiB |  95341 MiB |
|       from large pool |   8320 KiB |  19347 MiB |  94947 MiB |

In [8]:
import subprocess
import os
import time
command = [
    "vllm", "serve",
    "LORA_THE_BEST",
    "--host", "0.0.0.0",
    "--port", "8001",
    "--api-key", "7b956701760dccad68ef20684d635bd3ee82e2e8fa6287f048b572169f178d51",
    "--served-model-name", "LORA_THE_BEST",
    "--max-model-len", "8192",
    "--max-num-seqs", "4"
]

log_file = open("vllm_server.log", "w")

process = subprocess.Popen(
    command,
    stdout=log_file,
    stderr=subprocess.STDOUT,   # объединяем stderr с stdout
    text=True,                  # работаем со строками
    start_new_session=True      # отделяем процесс, чтобы он не завершился при выходе скрипта
)
#time.sleep(40)

# Ждём строку "Application startup complete" в логе
ready = False
while not ready:
    time.sleep(1)
    with open("vllm_server.log", "r") as f:
        if "Application startup complete" in f.read():
            ready = True
            break


print(f"Сервер запущен, PID: {process.pid}, логи пишутся в vllm_server.log")



Сервер запущен, PID: 213334, логи пишутся в vllm_server.log


In [9]:
from openai import OpenAI
import time
client = OpenAI(
    base_url="http://localhost:8001/v1",
    api_key="7b956701760dccad68ef20684d635bd3ee82e2e8fa6287f048b572169f178d51"
)
import pandas as pd
from tqdm import tqdm

In [10]:
start_time = time.perf_counter()
response = client.chat.completions.create(
    model="LORA_THE_BEST",
    messages=[{"role": "user", "content": "Мой кариотип 46,XX,t(5;10)(q12;p22). Что это значит?"}],
    max_tokens=1600,
    temperature=0.7
    
)

end_time = time.perf_counter()

answer = response.choices[0].message.content

print("Ответ модели:")
print(answer)
print(f"\nВремя выполнения: {end_time - start_time:.3f} секунд")

Ответ модели:
Тип перестройки: сбалансированная реципрокная транслокация

1. Характеристика перестройки
У пациента выявлена реципрокная транслокация между хромосомами 5 и 10 с точками разрыва в областях q12 и p22 соответственно.
Это означает, что фрагмент хромосомы 5 (область 5q12) обменялся местами с фрагментом хромосомы 10 (область 10p22).
Поскольку материал не утрачен и не продублирован, кариотип сбалансированный — общее количество и состав генетического материала сохранены.

2. Фенотип носителя
У лиц со сбалансированными реципрокными транслокациями физическое и умственное развитие, как правило, нормальное.
Проблемы чаще проявляются в репродуктивной сфере (бесплодие/невынашивание), а не как системные клинические симптомы.

3. Репродуктивные последствия
У мужчин
• Возможна нормальная фертильность, однако часть сперматозоидов будет содержать несбалансированные хромосомные наборы из-за нарушенного спаривания хромосом в мейозе (формирование квадривалента).
• Возможны:
o повышенная доля 

In [ ]:
process.terminate()

: 